# SAM 2 for Roman Space Telescope: Tutorial 04
## Segment Everything in an Astronomical Image Using Fine-Tuned Checkpoints

This tutorial demonstrates how to take the **best checkpoint (`checkpoint_best.pt`)** resulting from your fine-tuning run (e.g. Tutorial 02 or Tutorial 03) and use it to perform **promptless automatic mask generation ("Segment Everything")** across an entire astronomical FITS image.

### Workflow Overview:
1. **Load Checkpoint**: Instantiate `SAM2Segmentor.from_checkpoint('checkpoint_best.pt')`.
2. **Load FITS Image**: Ingest an astronomical exposure with celestial WCS metadata.
3. **Segment Everything**: Run single-pass feature encoding and batched prompt query evaluation (`prompt_mode='astronomical'` or `'grid'`) with the fine-tuned mask decoder.
4. **Photometry Measurement**: Measure net flux, SNR, centroid coordinates, and half-light radius on all extracted masks.
5. **Diagnostic Visualizations**: Display publication-ready 3-panel detection summaries and colorful segmentation overlays with `leakySB` stretch.
6. **Export FITS**: Save the labeled masks into a multi-extension FITS file with a standard `SEGMAP` HDU.

### Google Colab Setup (Optional)
If running in Google Colab, uncomment and run the setup cell below:

In [ ]:
# In Google Colab, uncomment and run:
# !git clone https://github.com/PabloMSanAla/sam2roman.git
# %cd sam2roman
# !pip install -e .
#
# If your FITS images and checkpoints are on Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

### Step 1: Load Fine-Tuned Weights into `SAM2Segmentor`

We use `SAM2Segmentor.from_checkpoint()`. It automatically reads the base model architecture (e.g. `facebook/sam2.1-hiera-tiny`), restores the fine-tuned prompt encoder and mask decoder weights, and sets the model to evaluation mode.

In [ ]:
import os
import sys
from pathlib import Path

for p in ['/content/sam2roman/src', 'src', '../src', '../../src']:
    resolved = Path(p).resolve()
    if resolved.exists() and str(resolved) not in sys.path:
        sys.path.insert(0, str(resolved))
        break

import matplotlib.pyplot as plt
import numpy as np
import torch

from sam2roman.io.fits import FITSImage
from sam2roman.inference.segmentor import SAM2Segmentor
from sam2roman.visualization.display import plot_detection_summary, plot_masks_overlay
from sam2roman.utils.synthetic import generate_synthetic_roman_fits

output_dir = Path("checkpoints")
best_checkpoint = output_dir / "checkpoint_best.pt"

if not best_checkpoint.exists():
    # Fallback check
    candidates = list(output_dir.glob("checkpoint_*.pt"))
    if candidates:
        best_checkpoint = sorted(candidates)[-1]
    else:
        raise FileNotFoundError(
            f"No checkpoint found at {best_checkpoint}. "
            "Run tutorial 02 or 03 first to fine-tune and save a checkpoint!"
        )

print(f"Loading fine-tuned model from: {best_checkpoint}")
segmentor = SAM2Segmentor.from_checkpoint(best_checkpoint)
print(f"Device:           {segmentor.model_wrapper.device}")
print(f"Foundation Model: {segmentor.model_wrapper.model_id}")

### Step 2: Load Target Astronomical FITS Image

In [ ]:
data_dir = Path("sample_data")
data_dir.mkdir(parents=True, exist_ok=True)
fits_path = data_dir / "sample_roman_train_field.fits"

if not fits_path.exists():
    fits_path = data_dir / "sample_roman_wfi.fits"

if not fits_path.exists():
    print("Generating sample astronomical image...")
    generate_synthetic_roman_fits(
        output_path=fits_path,
        shape=(512, 512),
        n_stars=40,
        n_galaxies=25,
        seed=42,
    )

astro_img = FITSImage.from_file(fits_path)
print(f"Loaded FITS Image: {fits_path.name}")
print(f"Dimensions:        {astro_img.shape[0]} x {astro_img.shape[1]} pixels")
print(f"Flux range:        min={astro_img.data.min():.2f}, max={astro_img.data.max():.2f}, median={np.median(astro_img.data):.2f}")

### Step 3: Run 'Segment Everything' with the Fine-Tuned Model

We run `segmentor.segment_everything()`. It encodes the full image once with the vision backbone, evaluates candidate prompt queries with the fine-tuned mask decoder, and deduplicates redundant detections using IoU suppression.

**Key Parameters:**
- `prompt_mode='astronomical'`: Places prompts on detected candidate peaks (fastest for astronomical images).
- `prompt_mode='grid'`: Evaluates a dense $N \times N$ grid of prompt queries (e.g. `points_per_side=32`).
- `stretch_type='leakysb'`: Surface brightness transformation with identical RGB channels.
- `pred_iou_thresh=0.60`: Rejects low-confidence masks.
- `min_mask_region_area=12`: Filters hot pixels and cosmic ray artifacts.

In [ ]:
results = segmentor.segment_everything(
    image=astro_img,
    prompt_mode="astronomical",  # or 'grid'
    pred_iou_thresh=0.60,
    min_mask_region_area=12,
    iou_overlap_thresh=0.70,
    stretch_type="leakysb",
    verbose=True,
)

print(f"\nTotal detected sources: {len(results)}")

### Step 4: Measure Astronomical Source Photometry

We compute source photometry (integrated flux, background subtraction, SNR, peak intensity, and half-light radius) on each mask extracted by the fine-tuned model.

In [ ]:
catalog = results.compute_photometry(astro_img, verbose=True)

if catalog:
    print(f"\nSample source catalog entry (Source #1):")
    for k, v in catalog[0].items():
        print(f"  -> {k}: {v}")
    
    brightest = max(catalog, key=lambda s: s["net_flux"])
    print(f"\nBrightest detection: Source #{brightest['source_id']} with Net Flux = {brightest['net_flux']:.1f} (SNR: {brightest['snr']:.1f})")

### Step 5: Visualizations & Overlay Plots

We generate two diagnostic visualizations:
1. **3-Panel Detection Summary**: Stretched flux, SAM 2 mask overlay, and labeled integer segmentation map.
2. **Color Mask Overlay**: Bounding boxes and centroids labeled on top of the `leakySB` image.

In [ ]:
# 1. 3-Panel Detection Summary
fig_summary = plot_detection_summary(
    image=astro_img,
    seg_result=results,
    title_prefix=f"Fine-Tuned SAM 2 ({astro_img.shape[0]}x{astro_img.shape[1]})",
)
plt.show()

# 2. Segmentation Mask Overlay
fig_overlay = plot_masks_overlay(
    image=astro_img,
    masks=results.masks,
    boxes=results.boxes,
    centroids=results.centroids,
    stretch_type="leakysb",
    title="Fine-Tuned SAM 2 Source Detections",
)
plt.show()

### Step 6: Export Masks to Multi-Extension FITS

We save the resulting segmentation masks into a multi-extension FITS file preserving all original telescope and WCS header keywords.

In [ ]:
export_fits_path = Path("checkpoints") / f"{fits_path.stem}_finetuned_segmap.fits"
astro_img.save_masks_to_fits(
    masks=results.masks,
    output_path=export_fits_path,
    overwrite=True,
)
print(f"Successfully exported masks to: {export_fits_path}")